In [1]:
import tifffile
import numpy as np
import os
from ome_types import from_xml, model
from ome_types.model.simple_types import PixelType
from datetime import datetime

# Research

In [3]:
def get_ome_metadata(filepath: str, image_array: np.ndarray) -> model.OME:
    """
    Reads OME-XML from a TIFF file or creates a new, valid OME object.

    Args:
        filepath: Path to the TIFF file.
        image_array: The corresponding image NumPy array, used to populate
                     metadata if a new object is created.

    Returns:
        An ome_types.model.OME object.
    """
    if os.path.exists(filepath):
        try:
            with tifffile.TiffFile(filepath) as tif:
                description = tif.pages[0].tags.get('ImageDescription')
                if description:
                    # Attempt to parse the XML from the description tag
                    return from_xml(description.value)
        except Exception as e:
            # If parsing fails, fall back to creating a new object
            print(f"⚠️ Could not parse existing metadata: {e}. Creating new OME object.")
            pass

    # --- Create a new, minimal OME object if one wasn't loaded ---
    print("✨ Creating a new, minimal OME metadata object.")
    ome = model.OME()
    ome.images.append(
        model.Image(
            id='Image:0',
            acquisition_date=datetime.now(),
            pixels=model.Pixels(
                id='Pixels:0',
                dimension_order='XY',
                size_c=1,
                size_t=1,
                size_z=1,
                size_y=image_array.shape[0],
                size_x=image_array.shape[1],
                type=PixelType(image_array.dtype.name),
                channels=[model.Channel(id='Channel:0:0')]
            )
        )
    )
    return ome

In [4]:
def save_image_with_ome(filepath: str, image: np.ndarray, ome: model.OME):
    """
    Saves an image and its OME-XML metadata to a compliant OME-TIFF file.

    Args:
        filepath: Path to save the output TIFF file.
        image: The image data as a NumPy array.
        ome: The OME metadata object to write.
    """
    # Convert the OME object to an XML string
    ome_xml_string = ome.to_xml()

    # Save as an OME-TIFF
    # ome=True ensures it's a valid OME-TIFF, not just a TIFF with XML
    tifffile.imwrite(
        filepath,
        image,
        photometric='minisblack',
        description=ome_xml_string,
        ome=True
    )
    print(f"✅ OME-TIFF and validated metadata successfully saved to '{filepath}'")

In [5]:
from pyimg import utils

In [ ]:
utils.read_metadata()

# examples

In [ ]:
>>> data = numpy.random.rand(2, 5, 3, 301, 219).astype('float32')
>>> imwrite(
...     'temp.tif',
...     data,
...     bigtiff=True,
...     photometric='rgb',
...     planarconfig='separate',
...     tile=(32, 32),
...     compression='zlib',
...     compressionargs={'level': 8},
...     predictor=True,
...     metadata={'axes': 'TZCYX'},
... )

In [ ]:
>>> volume = numpy.random.randn(6, 57, 256, 256).astype('float32')
>>> image_labels = [f'{i}' for i in range(volume.shape[0] * volume.shape[1])]
>>> imwrite(
...     'temp.tif',
...     volume,
...     imagej=True,
...     resolution=(1.0 / 2.6755, 1.0 / 2.6755),
...     metadata={
...         'spacing': 3.947368,
...         'unit': 'um',
...         'finterval': 1 / 10,
...         'fps': 10.0,
...         'axes': 'TZYX',
...         'Labels': image_labels,
...     },
... )

In [ ]:
>>> data = numpy.random.randint(0, 255, (8, 2, 512, 512, 3), 'uint8')
>>> subresolutions = 2
>>> pixelsize = 0.29  # micrometer
>>> with TiffWriter('temp.ome.tif', bigtiff=True) as tif:
...     metadata = {
...         'axes': 'TCYXS',
...         'SignificantBits': 8,
...         'TimeIncrement': 0.1,
...         'TimeIncrementUnit': 's',
...         'PhysicalSizeX': pixelsize,
...         'PhysicalSizeXUnit': 'µm',
...         'PhysicalSizeY': pixelsize,
...         'PhysicalSizeYUnit': 'µm',
...         'Channel': {'Name': ['Channel 1', 'Channel 2']},
...         'Plane': {'PositionX': [0.0] * 16, 'PositionXUnit': ['µm'] * 16},
...         'Description': 'A multi-dimensional, multi-resolution image',
...         'MapAnnotation': {  # for OMERO
...             'Namespace': 'openmicroscopy.org/PyramidResolution',
...             '1': '256 256',
...             '2': '128 128',
...         },
...     }
...     options = dict(
...         photometric='rgb',
...         tile=(128, 128),
...         compression='jpeg',
...         resolutionunit='CENTIMETER',
...         maxworkers=2,
...     )
...     tif.write(
...         data,
...         subifds=subresolutions,
...         resolution=(1e4 / pixelsize, 1e4 / pixelsize),
...         metadata=metadata,
...         **options,
...     )
...     # write pyramid levels to the two subifds
...     # in production use resampling to generate sub-resolution images
...     for level in range(subresolutions):
...         mag = 2 ** (level + 1)
...         tif.write(
...             data[..., ::mag, ::mag, :],
...             subfiletype=1,
...             resolution=(1e4 / mag / pixelsize, 1e4 / mag / pixelsize),
...             **options,
...         )
...     # add a thumbnail image as a separate series
...     # it is recognized by QuPath as an associated image
...     thumbnail = (data[0, 0, ::8, ::8] >> 2).astype('uint8')
...     tif.write(thumbnail, metadata={'Name': 'thumbnail'})

The OME schema is large and complex with references between elements etc.  
Tifffile only deals with Image/Pixels/Plane elements.  
Either provide your own OME-XML (imwrite(..., description=ome_xml, metadata=None))   
or edit the OME-XML created by tifffile using a dedicated library, e.g. ome_types, and write it back to the file:

In [ ]:
import numpy
import ome_types
from tifffile import imwrite, tiffcomment

filename = 'test.ome.tif'

imwrite(
    filename,
    numpy.random.randint(0, 1023, (4, 256, 256, 3), 'uint16'),
    bigtiff=True,
    photometric='RGB',
    tile=(64, 64),
    metadata={
        'axes': 'ZYXS',
        'SignificantBits': 10,
        'Plane': {'PositionZ': [0.0, 1.0, 2.0, 3.0]},
    },
)

ome_xml = tiffcomment(filename)
ome = ome_types.from_xml(ome_xml)
ome.images[0].description = 'Image 0 description'
ome_xml = ome.to_xml()
tiffcomment(filename, ome_xml)

In [ ]:
#You can pass a valid OME-XML byte string as description argument, 
imwrite(..., description=ome_xml, metadata=None)

In [ ]:
tifffile.OmeXml.validate

In [ ]:
from ome_types.model import OME, Image, Pixels

ome = OME()
img = Image(
    pixels=Pixels(
        size_c=2, size_t=1, size_x=512, size_y=512, size_z=15,
        type='uint16', dimension_order='XYZCT',
        metadata_only=True
    )
)
ome.images.append(img)

print(ome.to_xml())

In [ ]:
<OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd">
    <Image ID="Image:1">
        <Pixels DimensionOrder="XYZCT" ID="Pixels:12" SizeC="2" SizeT="1" SizeX="512" SizeY="512" SizeZ="15" Type="uint16">
            <MetadataOnly />
        </Pixels>
    </Image>
</OME>

Again: either provide an OME-XML as description or edit the minimal OME-XML created by tifffile using a dedicated XML or OME-XML library and write it back to the file. Why expect tifffile to support > 400 OME-XML elements and attributes with cross references plus Bio-Formats, OMERO, and custom extensions via a Python dict interface?

# Decision

After reading all that, my strategy will be:

(1) Extract OME-XML from image with ome_types  
(2) Modify  
(3) Pass as description with ome = False  

# Action

In [9]:
path_to_input = "/Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/inputs/TD_03_Exemplar001.ome.tif"

In [7]:
import ome_types

In [8]:
def read_metadata(input_path):
    ome = ome_types.from_tiff(input_path)
    return ome

In [10]:
ome = read_metadata(input_path=path_to_input)

In [11]:
ome

OME(
   images=[<1 field_type>],
   uuid='urn:uuid:b413ac24-f26b-11ee-bbfa-0242ac110002',
   creator='Ashlar v1.17.0',
)

In [12]:
with tifffile.TiffFile(path_to_input) as tif:
    if tif.ome_metadata:
        ome = ome_types.from_xml(tif.ome_metadata)

In [13]:
ome

OME(
   images=[<1 field_type>],
   uuid='urn:uuid:b413ac24-f26b-11ee-bbfa-0242ac110002',
   creator='Ashlar v1.17.0',
)

In [14]:
ome.images[0]

Image(
   id='Image:0',
   name='Image0',
   pixels={'channels': [{'light_path': {}, 'id': 'Channel:0:0', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:1', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:2', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:3', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:4', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:5', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:6', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:7', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:8', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:9', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:10', 'samples_per_pixel': 1}, {'light_path': {}, 'id': 'Channel:0:11', 'samples_per_pixel': 1}], 'tiff_data_blocks': [{'plane_count': 12}], 'id': 'Pixels:0', 'dimension_order': <Pixels_DimensionOrder.XYCZT: 'XYCZT'>, 'type': <PixelType.UINT16: 'uin

In [15]:
ome.images[0].pixels

Pixels(
   id='Pixels:0',
   channels=[<12 field_type>],
   tiff_data_blocks=[{'plane_count': 12}],
   dimension_order='XYCZT',
   type='uint16',
   size_x=2511,
   size_y=3139,
   size_z=1,
   size_c=12,
   size_t=1,
   physical_size_x=0.6499999761581421,
   physical_size_y=0.6499999761581421,
)

In [17]:
# lets image I bin 2x2 therefore phisical size and size_x and size_y should change

dask.array.coarsen (trim_excess=True) removes excess pixels that do not fit the kernel.

In [20]:
ome.images[0].pixels.size_x = ome.images[0].pixels.size_x // 2
ome.images[0].pixels.size_y = ome.images[0].pixels.size_y // 2
ome.images[0].pixels.physical_size_x = ome.images[0].pixels.physical_size_x * 2
ome.images[0].pixels.physical_size_y = ome.images[0].pixels.physical_size_y * 2

In [22]:
ome.images[0].pixels

Pixels(
   id='Pixels:0',
   channels=[<12 field_type>],
   tiff_data_blocks=[{'plane_count': 12}],
   dimension_order='XYCZT',
   type='uint16',
   size_x=1255,
   size_y=1569,
   size_z=1,
   size_c=12,
   size_t=1,
   physical_size_x=1.2999999523162842,
   physical_size_y=1.2999999523162842,
)

# run test, load image and save image

In [23]:
import dask.array as da
import tifffile
import numpy as np
from dask_image import imread
import ome_types
import argparse
from loguru import logger
from ome_types import model

In [46]:
path_to_input = "/Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/inputs/TD_03_Exemplar001.ome.tif"
output_path = "/Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/outputs/TD_03_Exemplar001_bin2x2_v2.ome.tif"

In [43]:
image = imread.imread(path_to_input).rechunk(chunks={0:1, 1:512, 2:512})
image.shape

(12, 3139, 2511)

In [ ]:
binning_factor = 2
axes_to_bin = {1: binning_factor, 2: binning_factor}
output_image = da.coarsen(np.mean, image, axes_to_bin, trim_excess=True)
output_image = output_image.round().astype(image.dtype)
# output_image_rounded = output_image_rounded.round().astype(image.dtype)

In [45]:
ome = ome_types.from_tiff(path_to_input)

ome.images[0].pixels.size_x = ome.images[0].pixels.size_x // 2
ome.images[0].pixels.size_y = ome.images[0].pixels.size_y // 2
ome.images[0].pixels.physical_size_x = ome.images[0].pixels.physical_size_x * 2
ome.images[0].pixels.physical_size_y = ome.images[0].pixels.physical_size_y * 2

ome_xml_bytes = ome_types.to_xml(ome).encode('utf-8')

In [47]:
tifffile.imwrite(
    output_path, 
    output_image, 
    bigtiff=True, 
    description=ome_xml_bytes
    )

In [48]:
type(ome)

ome_types._autogenerated.ome_2016_06.ome.OME

# Test read_metadata

## create array to demo no ome

In [50]:
import numpy as np

In [54]:
data = np.random.randint(0, 255, (8, 2, 512, 512, 3), 'uint8')

In [56]:
tifffile.imwrite("out.tif", data)


In [ ]:
for path

In [59]:
with tifffile.TiffFile("out.tif") as tif:
    ome_xml = tif.ome_metadata

In [62]:
type(ome_xml)

NoneType

In [64]:
try:
    ome = ome_types.from_tiff( path = "out.tif")
except ValueError as e:
    logger.warning(f"Could not parse OME-XML metadata. "
                    f"The file may not contain it or it's malformed.")
    logger.debug(f"Underlying error: {e}")




2025-08-11 11:32:23.774 | WARNING  | __main__:<module>:4 - Could not parse OME-XML metadata. The file may not contain it or it's malformed.
2025-08-11 11:32:23.775 | DEBUG    | __main__:<module>:6 - Underlying error: Could not parse XML from b'{"shape": [8, 2, 512, 512, 3]}'


In [71]:
pyramid_ome = ome_types.from_tiff("/Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/inputs/TD_03_Exemplar001.ome.tif")

In [72]:
describe_ome(pyramid_ome)

2025-08-11 11:38:58.018 | INFO     | __main__:describe_ome:18 - Pyramid structure not detected (or images have different names).
2025-08-11 11:38:58.018 | INFO     | __main__:describe_ome:21 - ==================== IMAGE DETAILS ====================
2025-08-11 11:38:58.019 | INFO     | __main__:describe_ome:26 - Processing Image 1/1
2025-08-11 11:38:58.019 | DEBUG    | __main__:describe_ome:36 - Pixels: ID='Pixels:0', DimensionOrder='Pixels_DimensionOrder.XYCZT', Type='PixelType.UINT16', Size(XYZCT)=(2511, 3139, 1, 12, 1)
2025-08-11 11:38:58.020 | DEBUG    | __main__:describe_ome:42 - Found 12 channel(s):
2025-08-11 11:38:58.020 | DEBUG    | __main__:describe_ome:51 -   - Channel: ID='Channel:0:0', Name='Unnamed', Color='white', SamplesPerPixel='1'
2025-08-11 11:38:58.021 | DEBUG    | __main__:describe_ome:51 -   - Channel: ID='Channel:0:1', Name='Unnamed', Color='white', SamplesPerPixel='1'
2025-08-11 11:38:58.021 | DEBUG    | __main__:describe_ome:51 -   - Channel: ID='Channel:0:2', N

In [70]:
describe_ome(ome)

2025-08-11 11:37:48.474 | INFO     | __main__:describe_ome:18 - Pyramid structure not detected (or images have different names).
2025-08-11 11:37:48.475 | INFO     | __main__:describe_ome:21 - ==================== IMAGE DETAILS ====================
2025-08-11 11:37:48.476 | INFO     | __main__:describe_ome:26 - Processing Image 1/1
2025-08-11 11:37:48.476 | DEBUG    | __main__:describe_ome:36 - Pixels: ID='Pixels:0', DimensionOrder='Pixels_DimensionOrder.XYCZT', Type='PixelType.UINT16', Size(XYZCT)=(1255, 1569, 1, 12, 1)
2025-08-11 11:37:48.476 | DEBUG    | __main__:describe_ome:42 - Found 12 channel(s):
2025-08-11 11:37:48.477 | DEBUG    | __main__:describe_ome:51 -   - Channel: ID='Channel:0:0', Name='Unnamed', Color='white', SamplesPerPixel='1'
2025-08-11 11:37:48.478 | DEBUG    | __main__:describe_ome:51 -   - Channel: ID='Channel:0:1', Name='Unnamed', Color='white', SamplesPerPixel='1'
2025-08-11 11:37:48.478 | DEBUG    | __main__:describe_ome:51 -   - Channel: ID='Channel:0:2', N

In [69]:
def describe_ome(ome: model.OME):
    """
    Logs the main features of an OME object using loguru.

    Args:
        ome: An ome_types.model.OME object.
    """
    if not ome.images:
        logger.warning("The OME object contains no images.")
        return


    # Check for pyramids (a common convention is multiple series/images)
    is_pyramid = len(ome.images) > 1 and all(img.name == ome.images[0].name for img in ome.images)
    if is_pyramid:
        logger.info(f"Image '{ome.images[0].name}' appears to be a pyramid with {len(ome.images)} levels.")
    else:
        logger.info("Pyramid structure not detected (or images have different names).")

    # Add a separator for clarity in the logs
    logger.info("=" * 20 + " IMAGE DETAILS " + "=" * 20)

    for i, image in enumerate(ome.images):
        # Use a logger context to add image-specific details to each log message
        with logger.contextualize(image_id=image.id, image_name=image.name):
            logger.info(f"Processing Image {i+1}/{len(ome.images)}")

            # --- PIXELS ---
            pixels = image.pixels
            pixels_info = (
                f"Pixels: ID='{pixels.id}', "
                f"DimensionOrder='{pixels.dimension_order}', "
                f"Type='{pixels.type}', "
                f"Size(XYZCT)=({pixels.size_x}, {pixels.size_y}, {pixels.size_z}, {pixels.size_c}, {pixels.size_t})"
            )
            logger.debug(pixels_info)

            # --- CHANNELS ---
            if not pixels.channels:
                logger.warning("No channels found for this image.")
            else:
                logger.debug(f"Found {len(pixels.channels)} channel(s):")
                for chan in pixels.channels:
                    # Format color for better readability
                    channel_info = (
                        f"  - Channel: ID='{chan.id}', "
                        f"Name='{chan.name or 'Unnamed'}', "
                        f"Color='{chan.color}', "
                        f"SamplesPerPixel='{chan.samples_per_pixel or 'N/A'}'"
                    )
                    logger.debug(channel_info)
    logger.info("=" * 20 + " END OF REPORT " + "=" * 20)

# Test final version of describe image and load metadata

In [ ]:
from pathlib import Path
import dask.array as da
import dask_image.imread
from loguru import logger
import ome_types

def describe_image(file_path: Path):
    """
    Calculates and logs basic features and metadata for a TIFF image
    using Dask for out-of-core computation.
    """
    logger.info(f"\n--- Describing: {file_path} ---")

    if not isinstance(file_path, Path):
        file_path = Path(file_path)

    allowed_extensions = ['.tif', '.tiff']
    if file_path.suffix.lower() not in allowed_extensions:
        logger.error(f"Unsupported file type: '{file_path.suffix}'. Please provide a .tif or .tiff file.")
        return

    try:
        # --- Section 1: File Structure (Pyramid Check) ---
        logger.info("\n## FILE STRUCTURE ##")
        with tifffile.TiffFile(str(file_path)) as tif:
            series = tif.series[0]
            is_pyramid = len(series.levels) > 1
            
            if is_pyramid:
                logger.info(f"Image is a pyramid")
                num_levels = len(tif.series[0].levels)
                logger.info(f"  - Levels: {num_levels}")
                # Log the shape of each pyramid level
                shapes = [level.shape for level in tif.series[0].levels]
                logger.info(f"  - Level Shapes: {shapes}")

        dask_arr = dask_image.imread.imread(str(file_path))
        logger.info(f"Successfully loaded image with Dask.")

        # OME METADATA
        try:
            ome = ome_types.from_tiff(file_path)
            for i, image in enumerate(ome.images):

                pixels = image.pixels
                phys_x = f"{pixels.physical_size_x} {pixels.physical_size_x_unit.value}" if pixels.physical_size_x else "N/A"
                pixels_info = (
                    f"  - Image [{i}]: '{image.name or 'Unnamed'}'\n"
                    f"    - Dimensions (Order: {pixels.dimension_order.value}): "
                    f"X={pixels.size_x}, Y={pixels.size_y}, Z={pixels.size_z}, C={pixels.size_c}, T={pixels.size_t}\n"
                    f"    - Pixel Type: {pixels.type.value}\n"
                    f"    - Pixel Size (X): {phys_x}"
                )
                logger.info(pixels_info)

                if  pixels.channels:
                    logger.info(f"    - Channels ({len(pixels.channels)}):")
                    for chan in pixels.channels:
                        color_hex = f"#{chan.color}" if chan.color else "N/A"
                        logger.info(f"      - Name: '{chan.name or 'Unnamed'}', Color: {color_hex}")

        except ValueError:
            # This is common if the file is a standard TIFF without OME-XML
            logger.info("-> No OME-XML metadata found in the file.")


        # --- Section 2: Dask Array Properties ---
        logger.info("\n## ARRAY PROPERTIES ##")
        logger.info(f"-> Shape: {dask_arr.shape}")
        logger.info(f"-> Data Type: {dask_arr.dtype}")
        logger.info(f"-> Dask Chunks: {dask_arr.chunksize}")
        logger.info(f"-> On-Disk Size: {dask_arr.nbytes / 1e9:.3f} GB")

        # --- Section 3: Pixel Statistics ---
        logger.info("\n## PIXEL STATISTICS ##")
        logger.info("-> Computing min, max, and mean...")
        min_val, max_val, mean_val = da.min(dask_arr), da.max(dask_arr), da.mean(dask_arr)
        
        # This triggers the actual computation
        min_res, max_res, mean_res = da.compute(min_val, max_val, mean_val)
        
        logger.info(f"-> Min Pixel Value: {min_res}")
        logger.info(f"-> Max Pixel Value: {max_res}")
        logger.info(f"-> Mean Pixel Value: {mean_res:.2f}")
        logger.info("-" * (len(file_path.name) + 18))

    except Exception as e:
        logger.info(f"Failed to process {file_path}: {e}")

In [82]:
describe_image("/Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/inputs/TD_03_Exemplar001.ome.tif")

2025-08-11 12:48:02.953 | INFO     | __main__:describe_image:12 - 
--- Describing: /Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/inputs/TD_03_Exemplar001.ome.tif ---
2025-08-11 12:48:02.954 | INFO     | __main__:describe_image:24 - 
## FILE STRUCTURE ##
2025-08-11 12:48:02.965 | INFO     | __main__:describe_image:31 -   - Levels: 3
2025-08-11 12:48:02.966 | INFO     | __main__:describe_image:34 -   - Level Shapes: [(12, 3139, 2511), (12, 1570, 1256), (12, 785, 628)]
2025-08-11 12:48:02.971 | INFO     | __main__:describe_image:37 - Successfully loaded image with Dask.
2025-08-11 12:48:02.979 | INFO     | __main__:describe_image:53 -   - Image [0]: 'Image0'
    - Dimensions (Order: XYCZT): X=2511, Y=3139, Z=1, C=12, T=1
    - Pixel Type: uint16
    - Pixel Size (X): 0.6499999761581421 µm
2025-08-11 12:48:02.979 | INFO     | __main__:describe_image:56 -     - Channels (12):
2025-08-11 12:48:02.980 | INFO     | __main__:describe_image:59 -       - Name: 'Unnamed', Color: #white
2